## 1. Import Libraries and Generate Data

In [1]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

np.random.seed(0)

# Generate synthetic data
n = 100
x = np.random.uniform(0, 10, n)
y = 3 * x + 5 + np.random.normal(0, 5, n)

## 2. Compute Basic Statistics and Fit Linear Regression

In [2]:
# Basic stats
print("Mean of x:", round(np.mean(x), 2))
print("Mean of y:", round(np.mean(y), 2))
print("Correlation:", round(np.corrcoef(x, y)[0,1], 2))

# Linear regression (manual formula)
slope = np.cov(x, y, bias=True)[0,1] / np.var(x)
intercept = np.mean(y) - slope * np.mean(x)

print("Slope:", round(slope, 2))
print("Intercept:", round(intercept, 2))

## 3. Visualize the Results

In [3]:
# Regression line
x_line = np.linspace(0, 10, 100)
y_line = slope * x_line + intercept

plt.figure(figsize=(8,4))

plt.subplot(1,2,1)
plt.scatter(x, y)
plt.plot(x_line, y_line, color='red')
plt.title("Regression")

plt.subplot(1,2,2)
plt.hist(y, bins=15)
plt.title("Distribution of y")

plt.tight_layout()
plt.show()

In [ ]:
!pip freeze

In [ ]:
!pip install

In [ ]:
import pandas as pd

df = pd.read_csv("customers.csv")
df.head()

In [ ]:
%%sql -r dataframe_3
CREATE OR REPLACE DATABASE DEMO_DB;
CREATE OR REPLACE SCHEMA DEMO_SCHEMA;

USE DATABASE DEMO_DB;
USE SCHEMA DEMO_SCHEMA;

CREATE OR REPLACE TABLE CUSTOMERS (
    id INT,
    first_name STRING,
    last_name STRING,
    email STRING,
    gender STRING,
    ip_address STRING
);

INSERT INTO CUSTOMERS (id, first_name, last_name, email, gender, ip_address)
VALUES
(1, 'Jennie', 'Chumley', 'jchumley0@ftc.gov', 'Female', '231.191.248.6'),
(2, 'Zacharie', 'Overal', 'zoveral1@slideshare.net', 'Male', '55.83.34.131'),
(3, 'Allard', 'Barmby', 'abarmby2@timesonline.co.uk', 'Male', '128.35.86.60'),
(4, 'Laurette', 'Dutt', 'ldutt3@reference.com', 'Female', '250.143.114.86'),
(5, 'Hansiain', 'Strowger', 'hstrowger4@hostgator.com', 'Male', '20.213.72.51');

In [ ]:
%%sql -r dataframe_4
SELECT * FROM CUSTOMERS;

In [ ]:
dataframe_4.head()

In [ ]:
import pandas as pd

# Copy the SQL output dataframe
df = dataframe_4.copy()

# Select relevant columns
df = df[[
    "FIRST_NAME",
    "LAST_NAME",
    "EMAIL",
    "GENDER",
    "IP_ADDRESS"
]]

# Example processing: Create a column for first name length category
df["NAME_LENGTH_TIER"] = pd.cut(
    df["FIRST_NAME"].str.len(),
    bins=[0, 4, 6, float("inf")],
    labels=["Short", "Medium", "Long"]
)

# Example processing: Extract email domain
df["EMAIL_DOMAIN"] = df["EMAIL"].str.split("@").str[1]

# Show the first few rows to make it available for SQL
df.head()

In [ ]:
%%sql -r dataframe_5
SELECT
    NAME_LENGTH_TIER,
    EMAIL_DOMAIN,
    COUNT(*) AS CUSTOMER_COUNT
FROM {{df}}
GROUP BY NAME_LENGTH_TIER, EMAIL_DOMAIN
ORDER BY NAME_LENGTH_TIER, CUSTOMER_COUNT DESC;

In [ ]:
%%sql -r dataframe_6
CREATE TABLE CUSTOMER_TRANSFORMED AS SELECT * FROM {{dataframe_5}};

In [ ]:
%%sql -r dataframe_7
SELECT * FROM CUSTOMER_TRANSFORMED;

In [ ]:
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark.functions import col, length, split, lit

# Get the active Snowpark session
session = get_active_session()

# Load the CUSTOMERS table into a Snowpark DataFrame
customers_df = session.table("CUSTOMERS")

# Process the data:
# - Compute length of the first name
# - Split the email by "@" using lit("@") and extract the domain
processed_df = (
    customers_df
    .with_column("FIRST_NAME_LENGTH", length(col("FIRST_NAME")))
    .with_column(
        "EMAIL_DOMAIN_ARRAY",
        split(col("EMAIL"), lit("@"))  # lit("@") is required for a string literal
    )
    .with_column(
        "EMAIL_DOMAIN",
        col("EMAIL_DOMAIN_ARRAY")[1]  # pick the second element from the split array
    )
    .select(
        "FIRST_NAME", "LAST_NAME", "EMAIL", "FIRST_NAME_LENGTH", "EMAIL_DOMAIN"
    )
)

# Show the result (first 5 rows as pandas DataFrame)
processed_df.limit(5).to_pandas()

In [ ]:
%%sql -r dataframe_1
DROP DATABASE DEMO_DB;